# Прогноз успешности 3D-печати детали

Простое базовое решение: загрузка данных, первичный анализ, решающее дерево глубины 4, сохранение `submission.csv`.

## 0. Библиотеки

In [105]:

import sys
from pathlib import Path

import optuna
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

SEED = 42

print("python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)

python       3.14.4
numpy        2.4.4
pandas       3.0.2
scikit-learn 1.8.0


## 1. Загрузка данных

Положите `train.csv`, `test.csv` и `sample_submission.csv` в папку с ноутбуком или укажите путь в `DATA_DIR`.

In [5]:
DATA_DIR = Path("data")  # папка с csv-файлами задачи

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:            ", train.shape)
print("test:             ", test.shape)
print("sample_submission:", sample_submission.shape)

train:             (16000, 21)
test:              (4000, 20)
sample_submission: (4000, 2)


In [6]:
train.head()

,id,material,printer_model,bed_surface,filament_color,nozzle_diameter_mm,nozzle_temp_c,bed_temp_c,print_speed_mm_s,layer_height_mm,...,model_mass_g,contact_area_cm2,geometry_complexity,overhang_angle_deg,estimated_time_min,filament_moisture_pct,cooling_percent,ambient_temp_c,hours_since_maintenance,target
0,115886,ABS,Model_A,textured,blue,0.4,245.4,107.4,35.6,0.32,...,66.2,44.2,0.762,46.2,101.6,NaN,0.0,33.2,357.0,1
1,118129,PLA,Model_C,PEI,natural,0.4,197.6,70.3,78.0,0.12,...,82.2,7.3,0.394,19.8,116.3,0.47,48.5,18.9,538.0,1
2,110735,ABS,Model_A,PEI,white,0.4,232.9,111.9,19.2,0.28,...,57.7,76.7,0.329,36.6,90.4,0.65,51.3,20.0,505.0,1
3,116662,ABS,Model_A,NaN,natural,0.4,224.0,101.1,37.5,0.12,...,34.2,44.4,0.628,33.5,141.5,1.02,3.4,20.8,272.0,1
4,117826,PETG,Model_D,PEI,red,0.4,268.1,93.6,71.7,0.16,...,147.7,19.5,0.361,29.2,199.5,0.47,62.5,21.2,79.0,0


Категориальные и числовые признаки. `id` в модель не передаём.

In [7]:
TARGET = "target"
ID = "id"
CAT_FEATURES = ["material", "printer_model", "bed_surface", "filament_color"]
NUM_FEATURES = [c for c in train.columns if c not in CAT_FEATURES + [ID, TARGET]]

assert list(test.columns) == [c for c in train.columns if c != TARGET]

print(f"Категориальные ({len(CAT_FEATURES)}):", CAT_FEATURES)
print(f"Числовые ({len(NUM_FEATURES)}):", NUM_FEATURES)

Категориальные (4): ['material', 'printer_model', 'bed_surface', 'filament_color']
Числовые (15): ['nozzle_diameter_mm', 'nozzle_temp_c', 'bed_temp_c', 'print_speed_mm_s', 'layer_height_mm', 'infill_percent', 'model_mass_g', 'contact_area_cm2', 'geometry_complexity', 'overhang_angle_deg', 'estimated_time_min', 'filament_moisture_pct', 'cooling_percent', 'ambient_temp_c', 'hours_since_maintenance']


## 2. Первичный анализ

### 2.1. Баланс классов

In [8]:
pd.DataFrame({
    "count": train[TARGET].value_counts().sort_index(),
    "share": train[TARGET].value_counts(normalize=True).sort_index().round(3),
})

,count,share
target,,
0,7207,0.45
1,8793,0.55


### 2.2. Пропуски

In [9]:
missing = pd.DataFrame({
    "train, %": train.isna().mean() * 100,
    "test, %": test.isna().mean() * 100,
}).round(2)
missing[missing["train, %"] > 0]

,"train, %","test, %"
ambient_temp_c,2.48,2.58
bed_surface,2.00,2.00
filament_moisture_pct,3.96,4.15
hours_since_maintenance,1.97,2.12


### 2.3. Описательная статистика числовых признаков

In [10]:
train[NUM_FEATURES].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
nozzle_diameter_mm,16000.0,0.48,0.12,0.40,0.40,0.40,0.60,0.80
nozzle_temp_c,16000.0,228.53,25.70,165.00,210.40,228.80,246.60,295.00
bed_temp_c,16000.0,72.79,25.69,20.00,54.40,71.90,90.80,135.00
print_speed_mm_s,16000.0,56.37,28.65,10.00,35.08,55.80,75.90,155.00
layer_height_mm,16000.0,0.23,0.09,0.10,0.16,0.24,0.28,0.48
infill_percent,16000.0,35.76,24.53,10.00,20.00,30.00,40.00,100.00
model_mass_g,16000.0,117.44,93.88,6.20,56.58,91.10,147.12,800.00
contact_area_cm2,16000.0,39.50,33.04,1.50,18.00,29.80,49.60,260.00
geometry_complexity,16000.0,0.50,0.22,0.02,0.33,0.50,0.67,0.98
overhang_angle_deg,16000.0,33.44,17.17,0.10,19.90,32.20,46.10,78.30


## 3. Подготовка признаков

- пропуски в числовых признаках → медиана;
- пропуски в категориальных признаках → категория `"missing"`;
- категориальные признаки → one-hot.

Предобработка в отдельной функции.

In [ ]:
def preprocessor():
    preprocessor = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), NUM_FEATURES),
            ("cat", make_pipeline(
                SimpleImputer(strategy="constant", fill_value="missing"),
                OneHotEncoder(handle_unknown="ignore", sparse_output=False), # sparse=False важно для GPU
            ), CAT_FEATURES),
        ],
        verbose_feature_names_out=False,
    )

    return preprocessor



X = train[CAT_FEATURES + NUM_FEATURES]
y = train[TARGET]
X_test = test[CAT_FEATURES + NUM_FEATURES]

## 4. Автоматический подбор гиперпараметров CatBoost (Optuna + GPU)

В этой ячейке запускается оптимизация гиперпараметров с использованием библиотеки Optuna. Используется GPU.

### Что подбирается:
* `iterations` — оптимальное количество деревьев.
* `learning_rate` — шаг градиентного спуска.
* `depth` — глубина деревьев.
* `l2_leaf_reg` — коэффициент регуляризации для борьбы с переобучением.
* `random_strength` и `bagging_temperature` — параметры случайности.

* **Целевая метрика для максимизации:** Macro F1-score на валидационной выборке.

In [ ]:
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

def objective(trial):
    params = {
        "task_type": "GPU",
        "iterations": trial.suggest_int("iterations", 1000, 3000, step=200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 5, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-1, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-2, 10.0, log=True),
        "bootstrap_type": "Bayesian",
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": SEED,
        "verbose": False
    }

    model = CatBoostClassifier(**params)
    
    model.fit(
        X_train_proc, y_train,
        eval_set=(X_valid_proc, y_valid),
        early_stopping_rounds=100, 
        verbose=False
    )

    preds = model.predict(X_valid_proc)
    return f1_score(y_valid, preds, average='macro')


X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

prep_instance = preprocessor()  
X_train_proc = prep_instance.fit_transform(X_train)
X_valid_proc = prep_instance.transform(X_valid)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=720)

# Печать лучших параметров
print("Лучшие параметры:", study.best_params)

## 5. Модель и валидация

Ансамбль catboost с подобранными гиперпараметрами.

In [ ]:
best_params = study.best_params
best_params.update({
    "task_type": "GPU",
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "verbose": 100  # Будет печатать лог каждые 100 итераций
})

best_classifier = CatBoostClassifier(**best_params)

model = Pipeline([
    ("prep", preprocessor()), 
    ("clf", best_classifier)
])

model.fit(
    X_train,
    y_train,
    clf__eval_set=(X_valid_proc, y_valid),
    clf__early_stopping_rounds=150
)

valid_pred = model.predict(X_valid)

print(f"macro F1 на валидации 1 яндексовские: {f1_score(y_valid, valid_pred, average='macro'):.4f}\n")
print(classification_report(y_valid, valid_pred, digits=3))


### Важность признаков

Признаки, которые использует дерево (с нулевой важностью не показаны).

In [101]:
feature_names = model.named_steps["prep"].get_feature_names_out()
importances = (
    pd.Series(model.named_steps["clf"].feature_importances_, index=feature_names, name="importance")
    .sort_values(ascending=False)
    .round(4)
)
importances[importances > 0].to_frame()

,importance
nozzle_temp_c,16.6191
bed_temp_c,11.9926
print_speed_mm_s,11.4697
cooling_percent,6.4593
layer_height_mm,6.1856
geometry_complexity,5.9398
model_mass_g,5.1003
material_PLA,4.9391
overhang_angle_deg,4.7665
filament_moisture_pct,4.1178


## 6. Обучение на всём train и предсказание для test

In [94]:
final_model = Pipeline([
    ("prep", preprocessor()), 
    ("clf", best_classifier)
])

final_model.fit(
    X_train,
    y_train,
    clf__eval_set=(X_valid_proc, y_valid),
    clf__early_stopping_rounds=150
)
test_pred = final_model.predict(X_test)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7737852	best: 0.7737852 (0)	total: 116ms	remaining: 3m 5s
100:	test: 0.9444715	best: 0.9444715 (100)	total: 12s	remaining: 2m 57s
200:	test: 0.9563005	best: 0.9563005 (200)	total: 21.3s	remaining: 2m 28s
300:	test: 0.9634618	best: 0.9635301 (294)	total: 31.3s	remaining: 2m 15s
400:	test: 0.9657015	best: 0.9657015 (400)	total: 39.7s	remaining: 1m 58s
500:	test: 0.9668378	best: 0.9668378 (500)	total: 46.9s	remaining: 1m 42s
600:	test: 0.9678225	best: 0.9678225 (600)	total: 53.4s	remaining: 1m 28s
700:	test: 0.9679661	best: 0.9681779 (664)	total: 59.5s	remaining: 1m 16s
800:	test: 0.9684344	best: 0.9684864 (773)	total: 1m 5s	remaining: 1m 5s
900:	test: 0.9686742	best: 0.9687575 (836)	total: 1m 10s	remaining: 54.9s
bestTest = 0.9687575102
bestIteration = 836
Shrink model to first 837 iterations.


## 7. Сохранение submission

In [102]:
submission = pd.DataFrame({ID: test[ID], TARGET: test_pred.astype(int)})

assert list(submission.columns) == ["id", "target"], "нужны ровно два столбца: id, target"
assert len(submission) == len(test), "число строк должно совпадать с test.csv"
assert submission[ID].is_unique, "id не должны повторяться"
assert set(submission[ID]) == set(sample_submission[ID]), "id должны совпадать с test.csv"
assert submission[TARGET].isin([0, 1]).all(), "target принимает только значения 0 и 1"

submission.to_csv("submission.csv", index=False)
print("Сохранено: submission.csv", submission.shape)
submission.head()

Сохранено: submission.csv (4000, 2)


,id,target
0,118423,1
1,101405,1
2,118315,0
3,105779,0
4,116242,1
